# Small Animal Classifier Sagemaker Serverless Deployment
This notebook deploys the [small-animal-classifier](https://github.com/agentmorris/small-animal-classifier/), trained and provided by Dan Morris, to a Sagemaker serverless endpoint. It is intended to be run in a SageMaker Notebook instance on the conda_pytorch_p10 kernel. This deployment uses the ONNX model stored in the Animl Model Zoo S3 bucket.

## Setup

In [6]:
import boto3
import sagemaker
from sagemaker import get_execution_role
import time
import json
import base64
from datetime import datetime

## Intialize AWS Session

In [7]:
session = boto3.Session()
sm = session.client('sagemaker')
region = session.region_name
account = boto3.client('sts').get_caller_identity().get('Account')

## Get IAM Role

Note: Ensure the IAM role has:

- `AmazonS3FullAccess`
- `AmazonSageMakerFullAccess`

In [8]:
role = sagemaker.get_execution_role()
print(f"Using role: {role}")

Using role: arn:aws:iam::830244800171:role/service-role/AmazonSageMaker-ExecutionRole-20210125T212674


## Create ECR Repository

In [9]:
# Create ECR repository if it doesn't exist
registry_name = "small-animal-classifier-sagemaker-serverless"
ecr = boto3.client('ecr')

try:
    ecr.create_repository(repositoryName=registry_name)
except ecr.exceptions.RepositoryAlreadyExistsException:
    print("ECR repository already exists")
    pass

ECR repository already exists


## Build and Upload Container

Builds the the inference container with small-animal-classifier and the sagemaker handler and uploads it to ECR. This step takes several minutes after it prints the 'Login Succeeded' message. Be patient and trust the process.

In [10]:
# flag to avoid timely image builds
should_create = True

if should_create:
    # Get auth token and login to ECR
    !aws ecr get-login-password --region {region} | docker login --username AWS --password-stdin {account}.dkr.ecr.{region}.amazonaws.com
    
    # Build container
    !docker build -q -t {registry_name} -f Dockerfile .
    
    # Tag and push to ECR
    image_uri = f"{account}.dkr.ecr.{region}.amazonaws.com/{registry_name}:latest"
    !docker tag {registry_name} {image_uri}
    !docker push {image_uri}
    
    print(f"Container pushed to: {image_uri}")

WARNING! Your password will be stored unencrypted in /home/ec2-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store

Login Succeeded
sha256:ebd39b32bbf22f8a71c2d150908e228f40f60b1db77df1fc27a42ecfade14acb
The push refers to repository [830244800171.dkr.ecr.us-west-2.amazonaws.com/small-animal-classifier-sagemaker-serverless]

bf18a086: Preparing 
8b86ea1b: Preparing 
b3da6508: Preparing 
a21bb0d2: Preparing 
4912fa17: Preparing 
a05b5092: Preparing 
e5121d04: Preparing 
df232f3c: Preparing 
b549523b: Preparing 
latest: digest: sha256:3748210f9012b438a01f8ad963a63f4929a0d4aaf88e9980d5c6f0d35a6bbe3b size: 2411
Container pushed to: 830244800171.dkr.ecr.us-west-2.amazonaws.com/small-animal-classifier-sagemaker-serverless:latest


## Create Sagemaker Model

In [20]:
model_prefix = "small-animal-classifier"

# Check if model already exists
model_already_created = False
for model_def in sm.list_models()['Models']:
    if model_prefix == model_def['ModelName']:
        create_model_response = model_def
        model_already_created = True

# Create model if it doesn't exist
if not model_already_created:
    create_model_response = sm.create_model(
        ModelName=model_prefix,
        ExecutionRoleArn=role,
        PrimaryContainer={
            "Image": image_uri,
            "ModelDataUrl": "s3://sagemaker-us-west-2-830244800171/small-animal-classifier/small-animal-classifier.tar.gz",
            "Environment": {
                "SAGEMAKER_PROGRAM": "serve.py"
            }
        }
    )

print(f"Model ARN: {create_model_response['ModelArn']}")

Model ARN: arn:aws:sagemaker:us-west-2:830244800171:model/small-animal-classifier


## Create Sagemaker Realtime Endpoint Configuration

In [21]:
# Create realtime and batch endpoint configuration
realtime_endpoint_config_name = f"{model_prefix}-realtime-config"

realtime_endpoint_config_response = sm.create_endpoint_config(
    EndpointConfigName=realtime_endpoint_config_name,
    ProductionVariants=[
        {
            "ModelName": model_prefix,
            "VariantName": "AllTraffic",
            "ServerlessConfig": {
                "MemorySizeInMB": 6144,  # 6GB memory
                "MaxConcurrency": 20       # Maximum concurrent invocations
            }
        }
    ]
)
print(f"Realtime endpoint config ARN: {realtime_endpoint_config_response['EndpointConfigArn']}")

Realtime endpoint config ARN: arn:aws:sagemaker:us-west-2:830244800171:endpoint-config/small-animal-classifier-realtime-config


## Create Realtime Endpoint

In [22]:
# Create realtime endpoint
realtime_endpoint_name = f"{model_prefix}-concurrency-20"
create_realtime_endpoint_response = sm.create_endpoint(
    EndpointName=realtime_endpoint_name,
    EndpointConfigName=realtime_endpoint_config_name
)

print(f"Endpoint ARN: {create_realtime_endpoint_response['EndpointArn']}")

# Wait for endpoint creation
resp = sm.describe_endpoint(EndpointName=realtime_endpoint_name)
realtime_status = resp['EndpointStatus']
print(f"Status: {realtime_status}")

while realtime_status == 'Creating':
    time.sleep(60)
    resp = sm.describe_endpoint(EndpointName=realtime_endpoint_name)
    realtime_status = resp['EndpointStatus']
    print(f"Status: {realtime_status}")
    if realtime_status == 'Failed':
        realtime_failure_reason = resp.get('FailureReason', 'No failure reason provided')
        print(f"Realtime endpoint deployment failed: {realtime_failure_reason}")
        break

# Get CloudWatch logs for the endpoint
logs = boto3.client('logs')

print(f"Realtime Arn: {resp['EndpointArn']}")
print(f"Realtime endpoint final status: {realtime_status}")
if realtime_status == 'Failed':
    realtime_log_group = f"/aws/sagemaker/Endpoints/{realtime_endpoint_name}"
    try:
        log_streams = logs.describe_log_streams(
            logGroupName=realtime_log_group,
            orderBy='LastEventTime',
            descending=True,
            limit=1
        )
        if log_streams['logStreams']:
            stream = log_streams['logStreams'][0]
            print(f"\nLog stream: {stream['logStreamName']}")
            realtime_events = logs.get_log_events(
                logGroupName=realtime_log_group,
                logStreamName=stream['logStreamName'],
                startFromHead=True
            )
            for event in realtime_events['events']:
                print(event['message'])
    except Exception as e:
        print(f"Error fetching logs: {str(e)}")

Endpoint ARN: arn:aws:sagemaker:us-west-2:830244800171:endpoint/small-animal-classifier-concurrency-20
Status: Creating
Status: Creating
Status: Creating
Status: Creating
Status: Creating
Status: Creating
Status: Creating
Status: Creating
Status: Creating
Status: InService
Realtime Arn: arn:aws:sagemaker:us-west-2:830244800171:endpoint/small-animal-classifier-concurrency-20
Realtime endpoint final status: InService


## Create Batch Endpoint Config

In [23]:
batch_endpoint_config_name = f"{model_prefix}-batch-config"

# Disable batch endpoint config creation if not needed
create_batch_endpoint_config = True

if create_batch_endpoint_config:
    batch_endpoint_config_response = sm.create_endpoint_config(
        EndpointConfigName=batch_endpoint_config_name,
        ProductionVariants=[
            {
                "ModelName": model_prefix,
                "VariantName": "AllTraffic",
                "ServerlessConfig": {
                    "MemorySizeInMB": 6144,  # 6GB memory
                    "MaxConcurrency": 80       # Maximum concurrent invocations
                }
            }
        ]
    )
    print(f"Batch endpoint config ARN: {batch_endpoint_config_response['EndpointConfigArn']}")

Batch endpoint config ARN: arn:aws:sagemaker:us-west-2:830244800171:endpoint-config/small-animal-classifier-batch-config


## Create Batch Endpoint

In [ ]:
# Create batch endpoint
create_batch_endpoint = True

if create_batch_endpoint:
    batch_endpoint_name = f"{model_prefix}-concurrency-80"
    create_batch_endpoint_response = sm.create_endpoint(
        EndpointName=batch_endpoint_name,
        EndpointConfigName=batch_endpoint_config_name
    )
    
    print(f"Endpoint ARN: {create_batch_endpoint_response['EndpointArn']}")
    
    # Wait for endpoint creation
    resp = sm.describe_endpoint(EndpointName=batch_endpoint_name)
    batch_status = resp['EndpointStatus']
    print(f"Status: {batch_status}")
    
    while batch_status == 'Creating':
        time.sleep(60)
        resp = sm.describe_endpoint(EndpointName=batch_endpoint_name)
        batch_status = resp['EndpointStatus']
        print(f"Status: {batch_status}")
        if batch_status == 'Failed':
            batch_failure_reason = resp.get('FailureReason', 'No failure reason provided')
            print(f"Batch endpoint deployment failed: {batch_failure_reason}")
            break
    
    # Get CloudWatch logs for the endpoint
    logs = boto3.client('logs')
    
    print(f"Batch Arn: {resp['EndpointArn']}")
    print(f"Batch endpoint final status: {batch_status}")
    if batch_status == 'Failed':
        batch_log_group = f"/aws/sagemaker/Endpoints/{batch_endpoint_name}"
        try:
            log_streams = logs.describe_log_streams(
                logGroupName=batch_log_group,
                orderBy='LastEventTime',
                descending=True,
                limit=1
            )
            if log_streams['logStreams']:
                stream = log_streams['logStreams'][0]
                print(f"\nLog stream: {stream['logStreamName']}")
                batch_events = logs.get_log_events(
                    logGroupName=batch_log_group,
                    logStreamName=stream['logStreamName'],
                    startFromHead=True
                )
                for event in batch_events['events']:
                    print(event['message'])
        except Exception as e:
            print(f"Error fetching logs: {str(e)}")

Endpoint ARN: arn:aws:sagemaker:us-west-2:830244800171:endpoint/small-animal-classifier-concurrency-80
Status: Creating
Status: Creating
